# Summarising docs


### I am using the following libraries: langchain-ollama (ChatOllama / OllamaLLM)
* LangChain
* langchain_ollama.OllamaEmbeddings(model="nomic-embed-text")
* langchain_chroma
* wget


### instead of: 


*   [`ibm-watsonx-ai`](https://ibm.github.io/watson-machine-learning-sdk/index.html) for using LLMs from IBM's watsonx.ai
*   [`LangChain`](https://www.langchain.com/) for using its different chain and prompt functions
*   [`Hugging Face`](https://huggingface.co/models?other=embeddings) and [`Hugging Face Hub`](https://huggingface.co/models?other=embeddings) for their embedding methods for processing text data
*   [`SentenceTransformers`](https://www.sbert.net/) for transforming sentences into high-dimensional vectors
*   [`Chroma DB`](https://www.trychroma.com/) for efficient storage and retrieval of high-dimensional text vector data
*   [`wget`](https://pypi.org/project/wget/) for downloading files from remote systems


### The installations

In [3]:
%%capture
%pip install -U \
langchain \
langchain-core \
langchain-community \
langchain-text-splitters \
langchain-huggingface \
langchain-chroma \
langchain-classic \
langchain-ollama \
chromadb \
sentence-transformers \
transformers \
huggingface-hub \
wget

In [4]:
%%capture
%pip install --upgrade numpy pandas scipy scikit-learn --upgrade-strategy only-if-needed

## All imports

In [9]:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings("ignore")

import wget

# LangChain imports
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_classic.chains import RetrievalQA, ConversationalRetrievalChain
from langchain_classic.prompts import PromptTemplate
from langchain_classic.memory import ConversationBufferMemory
from langchain_ollama import OllamaLLM

# Ollama (local bro)
print("All imports successful")

All imports successful


## Load

In [6]:
filename = 'companyPolicies.txt'
url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/6JDbUb_L3egv_eOkouY71A.txt'

# Getting the file
wget.download(url, out=filename)
print('file downloaded successfully')

file downloaded successfully


## Splitting the document into chunks-using LangChain

In [7]:
loader = TextLoader(filename)
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
texts = text_splitter.split_documents(documents)
print(len(texts), "chunks created from the document")

Created a chunk of size 1624, which is longer than the specified 1000
Created a chunk of size 1885, which is longer than the specified 1000
Created a chunk of size 1903, which is longer than the specified 1000
Created a chunk of size 1729, which is longer than the specified 1000
Created a chunk of size 1678, which is longer than the specified 1000
Created a chunk of size 2032, which is longer than the specified 1000
Created a chunk of size 1894, which is longer than the specified 1000


16 chunks created from the document


## Indexing Process: Embedding and Store

converting the chunks text into numbers
we create a default embedding model using Hugging Face and ingest into ChromaDB

In [8]:
embeddings = HuggingFaceEmbeddings()
docsearch = Chroma.from_documents(texts, embeddings) # stored emmbeddings in ChromaDB
print("document ingested")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3820.18it/s]


document ingested


## Retrieval

*  Model creation first

In [10]:
model_id = "mistral"

llm = OllamaLLM(
    model=model_id,
    temperature=0,
    num_predict=256,
)

## Let's integrate LangChain

In [11]:
qa = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=docsearch.as_retriever(), return_source_documents=False)
query = "What is mobile policy?"
qa.invoke(query)

{'query': 'What is mobile policy?',
 'result': ' The Mobile Policy is a set of guidelines that outlines the appropriate and responsible usage of mobile devices in an organization, ensuring compliance with company values, legal requirements, and best practices for security. It covers aspects such as acceptable use, security measures, confidentiality, cost management, compliance with laws and regulations, procedures for lost or stolen devices, and consequences for non-compliance. The policy aims to promote the responsible and secure use of mobile devices in line with legal and ethical standards.'}

In [12]:
query = "Can you summarize the document for me?"
qa.invoke(query)

{'query': 'Can you summarize the document for me?',
 'result': ' The document outlines the Code of Conduct, Health and Safety Policy, and Anti-discrimination and Harassment Policy of the organization. The Code of Conduct emphasizes integrity, respect, accountability, safety, and environmental responsibility as fundamental principles. It sets high ethical standards for all members, promoting honesty, transparency, diversity, and inclusivity.\n\nThe Health and Safety Policy focuses on prioritizing the well-being of employees, customers, and the public by diligently complying with health and safety laws and regulations. The goal is to maintain a workplace free from hazards, preventing accidents, injuries, and illnesses.\n\nLastly, the Anti-discrimination and Harassment Policy aims to create an environment that embraces diversity, values individual contributions, and prohibits discrimination, harassment, or any form of disrespectful behavior. It also outlines a recruitment policy that alig

## Done. Let's use prompt templates to guide the llm

In [13]:
query = "Can I eat in company vehicles?"
qa.invoke(query)

{'query': 'Can I eat in company vehicles?',
 'result': " The provided policies do not explicitly mention eating in company vehicles. However, since smoking and littering are prohibited in company vehicles, it can be inferred that there might be restrictions on food consumption as well to maintain cleanliness and prevent potential damage to the vehicles. It's best to consult with your supervisor or HR department for specific guidelines regarding eating in company vehicles."}

In [14]:
prompt_template = """Use the information from the document to answer the question at the end. If you don't know the answer, just say that you don't know, definitely do not try to make up an answer.

{context}

Question: {question}
"""
PROMPT = PromptTemplate(
    template=prompt_template, input_variables=["context", "question"]   
)

chain_type_kwargs = {"prompt": PROMPT}

In [15]:
query = "Can I eat in company vehicles?"
qa.invoke(query)

{'query': 'Can I eat in company vehicles?',
 'result': " The provided policies do not explicitly mention eating in company vehicles. However, since smoking and littering are prohibited in company vehicles, it can be inferred that there might be restrictions on food consumption as well to maintain cleanliness and prevent potential damage to the vehicles. It's best to consult with your supervisor or HR department for specific guidelines regarding eating in company vehicles."}

## Let's add some memory

In [16]:
query = "What I cannot do in it?"
qa.invoke(query)

{'query': 'What I cannot do in it?',
 'result': " Based on the provided policies, you cannot:\n\n1. Use company-provided internet and email services primarily for personal tasks during work hours (unless it doesn't interfere with work responsibilities).\n2. Harass, discriminate, or distribute offensive or inappropriate content through internet and email usage.\n3. Transmit sensitive company information via unsecured messaging apps or emails on mobile devices.\n4. Use mobile devices for personal charges on company-issued phones without keeping them separate from company accounts.\n5. Share login credentials for either the internet, email, or mobile devices with others.\n6. Ignore security concerns or suspicious activities related to your internet, email, or mobile device usage.\n7. Violate any relevant laws and regulations concerning internet, email, or mobile phone usage, including those related to copyright, data protection, privacy, etc."}

In [17]:
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)


qa = ConversationalRetrievalChain.from_llm(llm=llm, 
                                           chain_type="stuff", 
                                           retriever=docsearch.as_retriever(), 
                                           memory = memory, 
                                           get_chat_history=lambda h : h, 
                                           return_source_documents=False)

history = [] # stores the history of the conversation

In [18]:
query = "What is mobile policy?"
result = qa.invoke({"question":query}, {"chat_history": history})
print(result["answer"])

 The Mobile Policy is a set of guidelines that outlines the appropriate and responsible usage of mobile devices in an organization, ensuring compliance with company values, legal requirements, and best practices for security and cost management. It covers acceptable use, security measures, confidentiality, cost management, compliance with laws and regulations, procedures for lost or stolen devices, and consequences for non-compliance. The policy aims to promote the responsible and secure use of mobile devices in line with legal and ethical standards.


In [19]:
history.append((query, result["answer"]))

In [20]:
query = "List points in it?"
result = qa({"question": query}, {"chat_history": history})
print(result["answer"])

 The Mobile Phone Policy covers the following points:

1. Acceptable Use: Mobile devices are primarily intended for work-related tasks, with limited personal usage allowed.
2. Security: Safeguard your mobile device and access credentials, exercise caution when downloading apps or clicking links from unfamiliar sources, and promptly report security concerns or suspicious activities related to your mobile device.
3. Confidentiality: Avoid transmitting sensitive company information via unsecured messaging apps or emails, be discreet when discussing company matters in public spaces.
4. Cost Management: Keep personal phone usage separate from company accounts and reimburse the company for any personal charges on company-issued phones.
5. Compliance: Adhere to all pertinent laws and regulations concerning mobile phone usage, including those related to data protection and privacy.
6. Lost or Stolen Devices: Immediately report any lost or stolen mobile devices to the IT department or your supe

In [21]:
history.append((query, result["answer"]))

In [22]:
query = "What is the aim of it?"
result = qa({"question": query}, {"chat_history": history})
print(result["answer"])

 The objective of the Mobile Phone Policy is to ensure that employees utilize mobile phones in a manner consistent with company values, legal compliance, and ethical standards. It aims to promote responsible and secure use of mobile devices for work-related tasks while maintaining privacy, security, and cost management.


## Let's wrap it up and make an agent of it.

In [23]:
def qa():
    memory = ConversationBufferMemory(memory_key = "chat_history", return_message = True)
    qa = ConversationalRetrievalChain.from_llm(llm=llm, 
                                               chain_type="stuff", 
                                               retriever=docsearch.as_retriever(), 
                                               memory = memory, 
                                               get_chat_history=lambda h : h, 
                                               return_source_documents=False)
    history = []
    while True:
        query = input("Question: ")
        
        if query.lower() in ["quit","exit","bye"]:
            print("Answer: Goodbye!")
            break
            
        result = qa({"question": query}, {"chat_history": history})
        
        history.append((query, result["answer"]))
        
        print("Answer: ", result["answer"])

In [25]:
qa()

Answer: Goodbye!


In [26]:
%pip install gradio

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/30.7 MB ? eta -:--:--
    --------------------------------------- 0.5/30.7 MB 3.0 MB/s eta 0:00:11
   - -------------------------------------- 1.3/30.7 MB 2.8 MB/s eta 0:00:11
   -- ------------------------------------- 1.6/30.7 MB 2.8 MB/s eta 0:00:11
   -- ------------------------------------- 2.1/30.7 MB 2.4 MB/s eta 0:00:13
   --- ------------------------------------ 2.6/30.7 MB 2.6 MB/s eta 0:00:11
   ---- ----------------------------------- 3.1/30.7 MB 2.5 MB/s eta 0:00:11
   ---- ----------------------------------- 3.4/30.7 MB 2.5 MB/s eta 0:00:11
   ---- ----------------------------------- 3.7/30.7 MB 2.2 MB/s eta 0:00:13
   ----- ---------------------------------- 4.5/30.7 MB 2.3 MB/s eta 0:00:12
   ------ --------------------------------- 5.0/30.7 MB 2.3 MB/s eta 0:00:12
   ------- -------------------------------- 5.5/30.7 MB 2.3 MB/s eta 0:00:11
   --


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [27]:
import gradio as gr
def greet(name, intensity):
    return "Hello, " + name + "!" * int(intensity)
demo = gr.Interface(
    fn=greet,
    inputs=["text", "slider"],
    outputs="text"
)
demo.launch(server_name="127.0.0.1", server_port=7860)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Created dataset file at: .gradio\flagged\dataset1.csv


In [28]:
%pip install transformers
%pip install torch

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [30]:
import gradio as gr
from transformers import BlipProcessor, BlipForConditionalGeneration

processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")

def generate_caption(image):
    # Preprocess the image
    inputs = processor(images=image, return_tensors="pt")

    # Generate caption
    out = model.generate(**inputs)
    caption = processor.decode(out[0], skip_special_tokens=True)
    
    return caption

def caption_image(image):
    
    try:
        caption = generate_caption(image)
        return caption
    except Exception as e:
        return f"Error generating caption: {e}"
    
iface = gr.Interface(
    fn=caption_image,
    inputs=gr.Image(type="pil"),
    outputs="text",
    title="Image Captioning with BLIP",
    description="Upload an image and get a caption generated by the BLIP model."
)
iface.launch(server_name="127.0.0.1", server_port=7861)

Loading weights: 100%|██████████| 473/473 [00:00<00:00, 3061.52it/s]


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [32]:
%pip install torchvision

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/4.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.2 MB ? eta -:--:--
   ----- ---------------------------------- 0.5/4.2 MB 2.6 MB/s eta 0:00:02
   --------------- ------------------------ 1.6/4.2 MB 2.7 MB/s eta 0:00:01
   ----------------- ---------------------- 1.8/4.2 MB 2.5 MB/s eta 0:00:01
   ---------------------- ----------------- 2.4/4.2 MB 2.5 MB/s eta 0:00:01
   --------------------------- ------------ 2.9/4.2 MB 2.3 MB/s eta 0:00:01
   -------------------------------- ------- 3.4/4.2 MB 2.5 MB/s eta 0:00:01
   ------------------------------------- -- 3.9/4.2 MB 2.5 MB/s eta 0:00:01
   ---------------------------------------- 4.2/4.2 MB 2.4 MB/s  0:00:02
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [33]:
import torch
from torchvision.models import resnet18, ResNet18_Weights
model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1).eval()  # Set the model to evaluation mode

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\PC/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:21<00:00, 2.19MB/s]


## Let's define a predict function

In [38]:
import requests
from torchvision import transforms

# Download human-readable labels for ImageNet classes
response = requests.get("https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt")
labels = [l.strip() for l in response.text.split("\n") if l.strip()]

#Define image preprocessing for ResNet
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def predict(inp):
    # preprocess the image
    inp = transform(inp).unsqueeze(0)  # Add batch dimension
    # ensure model runs in inference mode
    with torch.no_grad():
        prediction = torch.nn.functional.softmax(model(inp)[0], dim=0)
        
    confidences = {
        labels[i]: float(prediction[i]) for i in range(len(labels))
    }
    
    return confidences

## Enter Gradio

In [47]:
gr.Interface(fn=predict,
             inputs=gr.Image(type="pil"),
             outputs=gr.Label(num_top_classes=3),
             title="Image Classification with ResNet-18 yekumamine",).launch(server_name="127.0.0.1", server_port=7867, share=True)

* Running on local URL:  http://127.0.0.1:7867
* Running on public URL: https://58ac8cb7b46cb9499e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
